In [123]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [124]:
from sklearn import set_config
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

In [125]:
data_path = r"..\data\interim\final_data.csv"

df = pd.read_csv(data_path,parse_dates=['tpep_pickup_datetime'])

df.sample(5)

,tpep_pickup_datetime,region,total_pickups,avg_pickups
111404,2016-03-09 11:00:00,12,4,3.0
196341,2016-02-13 05:15:00,22,38,89.0
169640,2016-02-08 02:00:00,19,16,18.0
54794,2016-01-25 18:30:00,6,36,33.0
110003,2016-02-23 20:45:00,12,3,4.0


In [126]:
df.head()

,tpep_pickup_datetime,region,total_pickups,avg_pickups
0,2016-01-01 00:00:00,0,197,197.0
1,2016-01-01 00:15:00,0,302,263.0
2,2016-01-01 00:30:00,0,326,295.0
3,2016-01-01 00:45:00,0,345,318.0
4,2016-01-01 01:00:00,0,333,324.0


In [127]:
# shape of the data

df.shape

(262080, 4)

In [128]:
# datatypes

df.dtypes

tpep_pickup_datetime    datetime64[ns]
region                           int64
total_pickups                    int64
avg_pickups                    float64
dtype: object

In [129]:
# checking null values

df.isnull().sum()

tpep_pickup_datetime    0
region                  0
total_pickups           0
avg_pickups             0
dtype: int64

In [130]:
# extract day of the week
df['day_of_week'] = df['tpep_pickup_datetime'].dt.day_of_week

# extract month 
df['month'] = df['tpep_pickup_datetime'].dt.month

df

,tpep_pickup_datetime,region,total_pickups,avg_pickups,day_of_week,month
0,2016-01-01 00:00:00,0,197,197.0,4,1
1,2016-01-01 00:15:00,0,302,263.0,4,1
2,2016-01-01 00:30:00,0,326,295.0,4,1
3,2016-01-01 00:45:00,0,345,318.0,4,1
4,2016-01-01 01:00:00,0,333,324.0,4,1
...,...,...,...,...,...,...
262075,2016-03-31 22:45:00,29,451,459.0,3,3
262076,2016-03-31 23:00:00,29,385,430.0,3,3
262077,2016-03-31 23:15:00,29,314,383.0,3,3
262078,2016-03-31 23:30:00,29,288,345.0,3,3


In [131]:
df.set_index('tpep_pickup_datetime',inplace=True)

In [132]:
df

,region,total_pickups,avg_pickups,day_of_week,month
tpep_pickup_datetime,,,,,
2016-01-01 00:00:00,0,197,197.0,4,1
2016-01-01 00:15:00,0,302,263.0,4,1
2016-01-01 00:30:00,0,326,295.0,4,1
2016-01-01 00:45:00,0,345,318.0,4,1
2016-01-01 01:00:00,0,333,324.0,4,1
...,...,...,...,...,...
2016-03-31 22:45:00,29,451,459.0,3,3
2016-03-31 23:00:00,29,385,430.0,3,3
2016-03-31 23:15:00,29,314,383.0,3,3


In [133]:
# create the region group

region_grp = df.groupby('region')

region_grp

In [134]:
# shifting periods

periods = list(range(1,5))

periods

[1, 2, 3, 4]

In [135]:
lag_features = region_grp['total_pickups'].shift(periods)

lag_features

,total_pickups_1,total_pickups_2,total_pickups_3,total_pickups_4
tpep_pickup_datetime,,,,
2016-01-01 00:00:00,NaN,NaN,NaN,NaN
2016-01-01 00:15:00,197.0,NaN,NaN,NaN
2016-01-01 00:30:00,302.0,197.0,NaN,NaN
2016-01-01 00:45:00,326.0,302.0,197.0,NaN
2016-01-01 01:00:00,345.0,326.0,302.0,197.0
...,...,...,...,...
2016-03-31 22:45:00,448.0,422.0,489.0,496.0
2016-03-31 23:00:00,451.0,448.0,422.0,489.0
2016-03-31 23:15:00,385.0,451.0,448.0,422.0


In [136]:
# merging with original df

data = pd.concat([df,lag_features],axis=1)

data

,region,total_pickups,avg_pickups,day_of_week,month,total_pickups_1,total_pickups_2,total_pickups_3,total_pickups_4
tpep_pickup_datetime,,,,,,,,,
2016-01-01 00:00:00,0,197,197.0,4,1,NaN,NaN,NaN,NaN
2016-01-01 00:15:00,0,302,263.0,4,1,197.0,NaN,NaN,NaN
2016-01-01 00:30:00,0,326,295.0,4,1,302.0,197.0,NaN,NaN
2016-01-01 00:45:00,0,345,318.0,4,1,326.0,302.0,197.0,NaN
2016-01-01 01:00:00,0,333,324.0,4,1,345.0,326.0,302.0,197.0
...,...,...,...,...,...,...,...,...,...
2016-03-31 22:45:00,29,451,459.0,3,3,448.0,422.0,489.0,496.0
2016-03-31 23:00:00,29,385,430.0,3,3,451.0,448.0,422.0,489.0
2016-03-31 23:15:00,29,314,383.0,3,3,385.0,451.0,448.0,422.0


In [137]:
df.shape,data.shape

((262080, 5), (262080, 9))

In [138]:
#  missing values

data.isnull().any(axis=1).sum()

np.int64(120)

In [139]:
data.iloc[8733:,:]

,region,total_pickups,avg_pickups,day_of_week,month,total_pickups_1,total_pickups_2,total_pickups_3,total_pickups_4
tpep_pickup_datetime,,,,,,,,,
2016-03-31 23:15:00,0,272,297.0,3,3,275.0,311.0,343.0,351.0
2016-03-31 23:30:00,0,275,288.0,3,3,272.0,275.0,311.0,343.0
2016-03-31 23:45:00,0,232,266.0,3,3,275.0,272.0,275.0,311.0
2016-01-01 00:00:00,1,210,243.0,4,1,NaN,NaN,NaN,NaN
2016-01-01 00:15:00,1,364,292.0,4,1,210.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2016-03-31 22:45:00,29,451,459.0,3,3,448.0,422.0,489.0,496.0
2016-03-31 23:00:00,29,385,430.0,3,3,451.0,448.0,422.0,489.0
2016-03-31 23:15:00,29,314,383.0,3,3,385.0,451.0,448.0,422.0


In [140]:
# dropping missing values - 120 rows

data.dropna(inplace=True)

In [141]:
data.isnull().any(axis=1).sum()

np.int64(0)

In [142]:
mapper = {name:f"lag_{ind+1}" for ind,name in enumerate(data.columns[5:])}

mapper

{'total_pickups_1': 'lag_1',
 'total_pickups_2': 'lag_2',
 'total_pickups_3': 'lag_3',
 'total_pickups_4': 'lag_4'}

In [143]:
data.rename(columns=mapper,inplace=True)

data

,region,total_pickups,avg_pickups,day_of_week,month,lag_1,lag_2,lag_3,lag_4
tpep_pickup_datetime,,,,,,,,,
2016-01-01 01:00:00,0,333,324.0,4,1,345.0,326.0,302.0,197.0
2016-01-01 01:15:00,0,329,326.0,4,1,333.0,345.0,326.0,302.0
2016-01-01 01:30:00,0,313,321.0,4,1,329.0,333.0,345.0,326.0
2016-01-01 01:45:00,0,271,301.0,4,1,313.0,329.0,333.0,345.0
2016-01-01 02:00:00,0,258,283.0,4,1,271.0,313.0,329.0,333.0
...,...,...,...,...,...,...,...,...,...
2016-03-31 22:45:00,29,451,459.0,3,3,448.0,422.0,489.0,496.0
2016-03-31 23:00:00,29,385,430.0,3,3,451.0,448.0,422.0,489.0
2016-03-31 23:15:00,29,314,383.0,3,3,385.0,451.0,448.0,422.0


In [144]:
# no. of rows in each month

data['month'].value_counts()

month
3    89280
1    89160
2    83520
Name: count, dtype: int64

In [145]:
# seperating jan and feb month data as training data

train_df = data[data['month'].isin([1,2])]

# saving train data
train_df_path = r"..\data\processed\train.csv"
train_df.to_csv(train_df_path,index=False)

train_df

,region,total_pickups,avg_pickups,day_of_week,month,lag_1,lag_2,lag_3,lag_4
tpep_pickup_datetime,,,,,,,,,
2016-01-01 01:00:00,0,333,324.0,4,1,345.0,326.0,302.0,197.0
2016-01-01 01:15:00,0,329,326.0,4,1,333.0,345.0,326.0,302.0
2016-01-01 01:30:00,0,313,321.0,4,1,329.0,333.0,345.0,326.0
2016-01-01 01:45:00,0,271,301.0,4,1,313.0,329.0,333.0,345.0
2016-01-01 02:00:00,0,258,283.0,4,1,271.0,313.0,329.0,333.0
...,...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,29,351,370.0,0,2,359.0,357.0,406.0,415.0
2016-02-29 23:00:00,29,248,321.0,0,2,351.0,359.0,357.0,406.0
2016-02-29 23:15:00,29,176,263.0,0,2,248.0,351.0,359.0,357.0


In [146]:
# seperating mar month data as testing data

test_df = data[data['month'].isin([3])]

# saving train data
test_df_path = r"..\data\processed\test.csv"
test_df.to_csv(test_df_path,index=False)

test_df

,region,total_pickups,avg_pickups,day_of_week,month,lag_1,lag_2,lag_3,lag_4
tpep_pickup_datetime,,,,,,,,,
2016-03-01 00:00:00,0,109,139.0,1,3,131.0,138.0,151.0,188.0
2016-03-01 00:15:00,0,84,117.0,1,3,109.0,131.0,138.0,151.0
2016-03-01 00:30:00,0,63,96.0,1,3,84.0,109.0,131.0,138.0
2016-03-01 00:45:00,0,54,79.0,1,3,63.0,84.0,109.0,131.0
2016-03-01 01:00:00,0,53,69.0,1,3,54.0,63.0,84.0,109.0
...,...,...,...,...,...,...,...,...,...
2016-03-31 22:45:00,29,451,459.0,3,3,448.0,422.0,489.0,496.0
2016-03-31 23:00:00,29,385,430.0,3,3,451.0,448.0,422.0,489.0
2016-03-31 23:15:00,29,314,383.0,3,3,385.0,451.0,448.0,422.0


In [147]:
# Splitting training data into X_train and y_train
X_train = train_df.drop(columns=['total_pickups'])
y_train = train_df[['total_pickups']]

# Splitting testing data into X_test and y_test
X_test = test_df.drop(columns=['total_pickups'])
y_test = test_df[['total_pickups']]

## Training a baseline model

In [148]:
set_config(transform_output='pandas')

In [149]:
# encoded data

encoder = ColumnTransformer(
    [
        ('OHE',OneHotEncoder(drop='first',sparse_output=False),['region','day_of_week'])
    ],
    remainder='passthrough',verbose_feature_names_out=False,n_jobs=-1
)

encoder

,transformers,"[('OHE', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,-1
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,False


In [150]:
# fitting on training data

encoder.fit_transform(X_train)

,region_1,region_2,region_3,region_4,region_5,region_6,region_7,region_8,region_9,region_10,...,day_of_week_3,day_of_week_4,day_of_week_5,day_of_week_6,avg_pickups,month,lag_1,lag_2,lag_3,lag_4
tpep_pickup_datetime,,,,,,,,,,,,,,,,,,,,,
2016-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,324.0,1,345.0,326.0,302.0,197.0
2016-01-01 01:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,326.0,1,333.0,345.0,326.0,302.0
2016-01-01 01:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,321.0,1,329.0,333.0,345.0,326.0
2016-01-01 01:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,301.0,1,313.0,329.0,333.0,345.0
2016-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,283.0,1,271.0,313.0,329.0,333.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,370.0,2,359.0,357.0,406.0,415.0
2016-02-29 23:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,321.0,2,351.0,359.0,357.0,406.0
2016-02-29 23:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,263.0,2,248.0,351.0,359.0,357.0


In [151]:
# train the model

lr = LinearRegression()

pipeline = Pipeline(
    [
        ('Encoder',encoder),
        ('Model',lr)
    ]
)

pipeline

,steps,"[('Encoder', ...), ('Model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('OHE', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,-1
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [152]:
# fit on training data

pipeline.fit(X_train,y_train)

,steps,"[('Encoder', ...), ('Model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('OHE', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,-1
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [153]:
# predicting on training data

y_pred_train = pipeline.predict(X_train)

y_pred_train

array([[358.90104769],
       [324.33860377],
       [306.06005841],
       ...,
       [185.87566753],
       [184.13185128],
       [179.54274769]], shape=(172680, 1))

In [154]:
# predicting on testing data

y_pred_test = pipeline.predict(X_test)

y_pred_train

array([[358.90104769],
       [324.33860377],
       [306.06005841],
       ...,
       [185.87566753],
       [184.13185128],
       [179.54274769]], shape=(172680, 1))

In [155]:
# evaluating the base line model

train_MAPE = mean_absolute_percentage_error(y_train,y_pred_train)

test_MAPE = mean_absolute_percentage_error(y_test,y_pred_test)

In [156]:
print(f"The trainig error is {train_MAPE:.2f}%")
print(f"The testing error is {test_MAPE:.2f}%")

The trainig error is 0.10%
The testing error is 0.09%


In [157]:
train_MAPE

0.09614103223195881

In [158]:
test_MAPE

0.09111219876863243